# Inventário Mobilidade — Engemap (protótipo) × Portal da Prefeitura de Apucarana — v1
**Somente leitura**: nada é alterado nos portais.
1. Lê o dashboard protótipo de Mobilidade no Portal Engemap e lista widgets, mapas, camadas, tabelas e campos.
2. Procura no Portal da Prefeitura (geo.apucarana.pr.gov.br) o que já está publicado sobre o mesmo tema.
3. Compara camada por camada (campos, nº de feições, ID_Rota) e diz o que já existe e o que falta.

Saída completa em `Documents\Saidas_Notebooks\Inventario_Mobilidade_Engemap_x_Apucarana_v1.txt` — é só arrastar para o chat.
Se aparecer uma caixa pedindo senha, digite a senha do usuário indicado e tecle Enter.

In [ ]:
# ===== CÉLULA 1 — Configuração =====
import os, sys, json, datetime, re, getpass, traceback, unicodedata

NB_NOME         = "Inventario_Mobilidade_Engemap_x_Apucarana_v1"
DASH_ID_ENGEMAP = "64f3842fab5e443b9a2e6f422b2e69f1"
URL_ENGEMAP     = "https://gisserver.engemap.com.br/portal"
USER_ENGEMAP    = "gisprofessional"
URL_APUC        = "https://geo.apucarana.pr.gov.br/portal"
USER_APUC       = "idepplan1"

# termos extras de busca no Portal da Prefeitura (além dos nomes encontrados no protótipo)
PALAVRAS = ["onibus", "ônibus", "bus", "Bus_line", "Bus_line2", "linha", "linhas", "horario", "horarios",
            "horários", "mobilidade", "transporte", "parada", "pontos de onibus", "Pontos_de_onibus",
            "itinerario", "CSV_Master", "sugestao", "sugestão", "terminal"]

home = os.path.expanduser("~")
_cands = [os.path.join(home, "Documents", "Saidas_Notebooks"),
          os.path.join(home, "OneDrive", "Documents", "Saidas_Notebooks"),
          os.path.join(home, "OneDrive", "Documentos", "Saidas_Notebooks"),
          os.path.join(home, "Documentos", "Saidas_Notebooks")]
PASTA_SAIDA = next((c for c in _cands if os.path.isdir(c)), _cands[0])
os.makedirs(PASTA_SAIDA, exist_ok=True)
PASTA_JSON = os.path.join(PASTA_SAIDA, NB_NOME + "_json")
os.makedirs(PASTA_JSON, exist_ok=True)
ARQ_TXT = os.path.join(PASTA_SAIDA, NB_NOME + ".txt")
with open(ARQ_TXT, "w", encoding="utf-8") as f:
    f.write(f"{NB_NOME} — executado em {datetime.datetime.now():%d/%m/%Y %H:%M:%S}\n\n")

def log(*a):
    s = " ".join(str(x) for x in a)
    print(s)
    with open(ARQ_TXT, "a", encoding="utf-8") as f:
        f.write(s + "\n")

def norm(s):
    s = unicodedata.normalize("NFKD", str(s or "")).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]", "", s)

def salvar_json(nome, obj):
    p = os.path.join(PASTA_JSON, nome)
    with open(p, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    return p

LINKS = []
log("▶ CÉLULA 1 — Configuração: iniciando...")
log("  Saída (.txt):", ARQ_TXT)
log("  Backups JSON:", PASTA_JSON)
log("✔ CÉLULA 1 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 2 — Conexão com o Portal da Prefeitura (Apucarana) =====
log("\n▶ CÉLULA 2 — Conexão com o Portal da Prefeitura de Apucarana: iniciando...")
from arcgis.gis import GIS
import arcpy
try:
    ATIVO = arcpy.GetActivePortalURL() or ""
except Exception:
    ATIVO = ""
log("  Portal ativo no ArcGIS Pro:", ATIVO or "(nenhum)")
gis_apuc = None
if "apucarana" in ATIVO.lower():
    try:
        gis_apuc = GIS("pro")
    except Exception as e:
        log("  Não consegui usar o login do ArcGIS Pro:", e)
if gis_apuc is None:
    log(f"  Digite a senha do usuário {USER_APUC} (Portal da Prefeitura) na caixa abaixo.")
    gis_apuc = GIS(URL_APUC, USER_APUC, getpass.getpass(f"Senha {USER_APUC} @ Apucarana: "))
_me = gis_apuc.users.me
log("  Conectado como:", _me.username if _me else "anônimo", "|", gis_apuc.url)
log("✔ CÉLULA 2 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 3 — Conexão com o Portal Engemap e leitura do dashboard protótipo =====
log("\n▶ CÉLULA 3 — Portal Engemap / dashboard protótipo: iniciando...")
gis_eng, dash, ENG_ANONIMO = None, None, False
if "engemap" in ATIVO.lower():
    try:
        gis_eng = GIS("pro"); dash = gis_eng.content.get(DASH_ID_ENGEMAP)
    except Exception as e:
        log("  Login do ArcGIS Pro não serviu para a Engemap:", e)
if dash is None:
    try:
        gis_eng = GIS(URL_ENGEMAP); dash = gis_eng.content.get(DASH_ID_ENGEMAP)
        if dash is not None:
            ENG_ANONIMO = True
            log("  Dashboard acessível sem login (público).")
    except Exception as e:
        log("  Acesso anônimo à Engemap falhou:", e)

def login_engemap():
    global gis_eng, ENG_ANONIMO
    log(f"  Digite a senha do usuário {USER_ENGEMAP} (Portal Engemap) na caixa abaixo.")
    gis_eng = GIS(URL_ENGEMAP, USER_ENGEMAP, getpass.getpass(f"Senha {USER_ENGEMAP} @ Engemap: "))
    ENG_ANONIMO = False

if dash is None:
    login_engemap()
    dash = gis_eng.content.get(DASH_ID_ENGEMAP)
if dash is None:
    raise RuntimeError("Não encontrei o dashboard " + DASH_ID_ENGEMAP + " no Portal Engemap.")

dash_json = dash.get_data()
p = salvar_json("dashboard_prototipo_engemap.json", dash_json)
log(f"  Dashboard: {dash.title}  | tipo: {dash.type} | dono: {dash.owner} | acesso: {dash.access}")
log(f"  Modificado em: {datetime.datetime.fromtimestamp(dash.modified/1000):%d/%m/%Y %H:%M}")
log("  Backup do JSON do dashboard:", p)
LINKS.append(("Dashboard protótipo (Engemap)", f"{URL_ENGEMAP}/apps/dashboards/{DASH_ID_ENGEMAP}"))
LINKS.append(("Item do dashboard protótipo (Engemap)", f"{URL_ENGEMAP}/home/item.html?id={DASH_ID_ENGEMAP}"))
log("✔ CÉLULA 3 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 4 — Inventário do protótipo (widgets, mapas, camadas, campos) =====
log("\n▶ CÉLULA 4 — Inventário do protótipo: iniciando...")
from arcgis.features import FeatureLayerCollection
RE_ID = re.compile(r"\b[0-9a-f]{32}\b")

def limpar(s):
    return re.sub(r"<[^>]+>", " ", str(s or "")).strip()[:80]

# 4.1 — widgets e IDs referenciados
WIDGETS, IDS_POR_WIDGET = [], {}
def varrer(obj, rotulo):
    if isinstance(obj, dict):
        t = obj.get("type")
        if isinstance(t, str) and (t.endswith("Widget") or t.endswith("Selector")) and "id" in obj:
            rotulo = f"{t} | {limpar(obj.get('caption') or obj.get('name') or obj.get('title') or obj.get('id'))}"
            WIDGETS.append(rotulo)
        for k, v in obj.items():
            if k in ("itemId", "webmapId") and isinstance(v, str) and RE_ID.fullmatch(v):
                IDS_POR_WIDGET.setdefault(v, set()).add(rotulo)
            varrer(v, rotulo)
    elif isinstance(obj, list):
        for x in obj:
            varrer(x, rotulo)
varrer(dash_json, "(geral)")
outros = set(RE_ID.findall(json.dumps(dash_json))) - set(IDS_POR_WIDGET) - {DASH_ID_ENGEMAP}
for i in outros:
    IDS_POR_WIDGET.setdefault(i, set()).add("(citado no JSON — ex.: expressão Arcade)")

log(f"\n  4.1 Widgets/seletores encontrados: {len(WIDGETS)}")
for w in WIDGETS:
    log("   -", w)
dexp = dash_json.get("dataExpressions") or []
if dexp:
    log(f"  Expressões de dados Arcade: {len(dexp)}")
    for d in dexp:
        log("   -", limpar(d.get("title") or d.get("name") or d.get("id")))

# 4.2 — resolver itens no Portal Engemap
def get_eng(iid):
    it = None
    try:
        it = gis_eng.content.get(iid)
    except Exception:
        pass
    if it is None and ENG_ANONIMO:
        login_engemap()
        try:
            it = gis_eng.content.get(iid)
        except Exception:
            pass
    return it

def detalhar_servico(item_ou_url, portal):
    camadas = []
    try:
        if isinstance(item_ou_url, str):
            flc = FeatureLayerCollection(item_ou_url, portal)
        else:
            flc = FeatureLayerCollection.fromitem(item_ou_url)
        for tipo, lista in (("Camada", flc.layers or []), ("Tabela", flc.tables or [])):
            for lyr in lista:
                pr = lyr.properties
                try:
                    n = lyr.query(where="1=1", return_count_only=True)
                except Exception:
                    n = "?"
                camadas.append({"tipo": tipo, "id": pr.get("id"), "nome": pr.get("name"),
                                "geometria": pr.get("geometryType", "-"), "n": n, "url": lyr.url,
                                "campos": [f["name"] for f in (pr.get("fields") or [])]})
    except Exception as e:
        camadas.append({"erro": str(e)})
    return camadas

FONTES_ENG = {}   # id -> dict
fila, vistos = list(IDS_POR_WIDGET), set()
while fila:
    iid = fila.pop(0)
    if iid in vistos:
        continue
    vistos.add(iid)
    it = get_eng(iid)
    if it is None:
        FONTES_ENG[iid] = {"titulo": "(não encontrado/sem permissão)", "tipo": "?", "camadas": [], "usado_em": sorted(IDS_POR_WIDGET.get(iid, []))}
        continue
    reg = {"titulo": it.title, "tipo": it.type, "dono": it.owner, "url": it.url, "acesso": it.access,
           "usado_em": sorted(IDS_POR_WIDGET.get(iid, ["(dentro de web map)"])), "camadas": [], "mapa": []}
    if it.type == "Web Map":
        wm = it.get_data()
        salvar_json(f"webmap_engemap_{iid}.json", wm)
        for grupo in ("operationalLayers", "tables"):
            for ol in wm.get(grupo, []) or []:
                subs = ol.get("layers") if ol.get("layerType") == "GroupLayer" else [ol]
                for s in subs or []:
                    pop = (s.get("popupInfo") or {})
                    reg["mapa"].append({"titulo": s.get("title"), "url": s.get("url"), "itemId": s.get("itemId"),
                                        "arcade_popup": len(pop.get("expressionInfos") or []),
                                        "tipo": "tabela" if grupo == "tables" else s.get("layerType")})
                    if s.get("itemId"):
                        IDS_POR_WIDGET.setdefault(s["itemId"], set()).add(f"web map: {it.title}")
                        fila.append(s["itemId"])
    elif it.type in ("Feature Service", "Map Service"):
        reg["camadas"] = detalhar_servico(it, gis_eng)
    FONTES_ENG[iid] = reg

log(f"\n  4.2 Itens usados pelo protótipo: {len(FONTES_ENG)}")
for iid, r in FONTES_ENG.items():
    log(f"\n  ■ {r['titulo']}  [{r['tipo']}]  id={iid}")
    log(f"    dono: {r.get('dono','?')} | acesso: {r.get('acesso','?')}")
    log(f"    usado em: {'; '.join(r['usado_em'])}")
    for m in r.get("mapa", []):
        log(f"    · (no mapa) {m['titulo']} [{m['tipo']}] item={m['itemId']} | popups Arcade: {m['arcade_popup']}")
    for c in r.get("camadas", []):
        if "erro" in c:
            log("    ! erro ao ler serviço:", c["erro"]); continue
        tem_id_rota = "SIM" if any(norm(x) == "idrota" for x in c["campos"]) else "não"
        log(f"    · {c['tipo']} {c['id']}: {c['nome']} | geom: {c['geometria']} | feições/linhas: {c['n']} | ID_Rota: {tem_id_rota}")
        log(f"      campos: {', '.join(c['campos'])}")
salvar_json("inventario_prototipo_engemap.json", FONTES_ENG)
log("\n✔ CÉLULA 4 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 5 — Busca do que já existe no Portal da Prefeitura =====
log("\n▶ CÉLULA 5 — Busca no Portal da Prefeitura de Apucarana: iniciando...")
termos = set(PALAVRAS)
for r in FONTES_ENG.values():
    if r["titulo"] and not r["titulo"].startswith("("):
        termos.add(r["titulo"])
    for c in r.get("camadas", []):
        if c.get("nome"):
            termos.add(c["nome"])
    for m in r.get("mapa", []):
        if m.get("titulo"):
            termos.add(m["titulo"])

TIPOS_UTEIS = {"Feature Service", "Map Service", "Web Map", "Dashboard", "Web Experience", "Form",
               "CSV", "Microsoft Excel", "Shapefile", "File Geodatabase", "Table", "StoryMap", "Web Mapping Application"}
CANDIDATOS = {}
for t in sorted(termos, key=str.lower):
    q = f'title:"{t}"' if " " in t else f"title:{t}"
    try:
        res = gis_apuc.content.search(query=q, max_items=100, outside_org=False)
    except Exception as e:
        log(f"  ! busca '{t}' falhou: {e}"); continue
    for it in res:
        if it.type in TIPOS_UTEIS:
            CANDIDATOS.setdefault(it.id, {"item": it, "termos": set()})["termos"].add(t)
log(f"  Termos pesquisados: {len(termos)} | itens candidatos encontrados: {len(CANDIDATOS)}")

INV_APUC = {}
for iid, d in sorted(CANDIDATOS.items(), key=lambda kv: (kv[1]["item"].type, kv[1]["item"].title.lower())):
    it = d["item"]
    reg = {"titulo": it.title, "tipo": it.type, "dono": it.owner, "acesso": it.access, "url": it.url,
           "modificado": f"{datetime.datetime.fromtimestamp(it.modified/1000):%d/%m/%Y}",
           "termos": sorted(d["termos"]), "camadas": [], "mapa": []}
    if it.type in ("Feature Service", "Map Service"):
        reg["camadas"] = detalhar_servico(it, gis_apuc)
    elif it.type == "Web Map":
        try:
            wm = it.get_data()
            salvar_json(f"webmap_apucarana_{iid}.json", wm)
            for grupo in ("operationalLayers", "tables"):
                for ol in wm.get(grupo, []) or []:
                    reg["mapa"].append({"titulo": ol.get("title"), "itemId": ol.get("itemId"), "url": ol.get("url")})
        except Exception as e:
            reg["mapa"].append({"titulo": f"(erro: {e})"})
    INV_APUC[iid] = reg
    log(f"\n  ■ {it.title}  [{it.type}]  id={iid}")
    log(f"    dono: {it.owner} | acesso: {it.access} | modificado: {reg['modificado']} | achado por: {', '.join(reg['termos'])}")
    for m in reg["mapa"]:
        log(f"    · (no mapa) {m.get('titulo')} | item={m.get('itemId')}")
    for c in reg["camadas"]:
        if "erro" in c:
            log("    ! erro ao ler serviço:", c["erro"]); continue
        tem_id_rota = "SIM" if any(norm(x) == "idrota" for x in c["campos"]) else "não"
        log(f"    · {c['tipo']} {c['id']}: {c['nome']} | geom: {c['geometria']} | feições/linhas: {c['n']} | ID_Rota: {tem_id_rota}")
        log(f"      campos: {', '.join(c['campos'])}")
salvar_json("inventario_portal_apucarana.json", INV_APUC)
log("\n✔ CÉLULA 5 concluída com sucesso.")


In [ ]:
# ===== CÉLULA 6 — Comparação camada a camada + links =====
log("\n▶ CÉLULA 6 — Comparação protótipo × Portal da Prefeitura: iniciando...")
camadas_apuc = []
for iid, r in INV_APUC.items():
    for c in r["camadas"]:
        if "erro" not in c:
            camadas_apuc.append((iid, r, c))

def parecidas(nome):
    n = norm(nome)
    ex = [x for x in camadas_apuc if norm(x[2]["nome"]) == n]
    if ex:
        return ex
    return [x for x in camadas_apuc if n and (n in norm(x[2]["nome"]) or norm(x[2]["nome"]) in n)]

RESUMO = []
log("\n  ===== QUADRO-RESUMO =====")
for iid, r in FONTES_ENG.items():
    for c in r.get("camadas", []):
        if "erro" in c:
            continue
        achados = parecidas(c["nome"])
        if not achados:
            status = "NÃO ENCONTRADA no Portal da Prefeitura"
            log(f"\n  ✗ {c['nome']} ({c['tipo']}, {c['n']} feições) → {status}")
            RESUMO.append({"camada_prototipo": c["nome"], "status": status})
            continue
        log(f"\n  ✓ {c['nome']} ({c['tipo']}, {c['n']} feições no protótipo) → {len(achados)} candidata(s):")
        for aid, ar, ac in achados:
            faltam = [x for x in c["campos"] if norm(x) not in {norm(y) for y in ac["campos"]}]
            sobram = [x for x in ac["campos"] if norm(x) not in {norm(y) for y in c["campos"]}]
            status = "campos IGUAIS" if not faltam and not sobram else "campos DIFERENTES"
            log(f"     - '{ac['nome']}' no item '{ar['titulo']}' (id={aid}, dono={ar['dono']}, acesso={ar['acesso']})")
            log(f"       feições: protótipo={c['n']} × Prefeitura={ac['n']} | {status}")
            if faltam: log(f"       faltam na Prefeitura: {', '.join(faltam)}")
            if sobram: log(f"       só existem na Prefeitura: {', '.join(sobram)}")
            RESUMO.append({"camada_prototipo": c["nome"], "item_apucarana": aid, "camada_apucarana": ac["nome"],
                           "n_prototipo": c["n"], "n_apucarana": ac["n"], "status": status,
                           "faltam": faltam, "sobram": sobram})

# itens de mobilidade que só existem na Prefeitura (apps, mapas, formulários)
log("\n  ===== ITENS DE APP/MAPA/FORMULÁRIO NA PREFEITURA =====")
for iid, r in INV_APUC.items():
    if r["tipo"] in ("Web Map", "Dashboard", "Web Experience", "Form", "StoryMap", "Web Mapping Application"):
        log(f"   - {r['titulo']} [{r['tipo']}] id={iid} | dono={r['dono']} | acesso={r['acesso']}")

salvar_json("resumo_comparacao.json", RESUMO)

for iid, r in INV_APUC.items():
    LINKS.append((f"Prefeitura — {r['titulo']} [{r['tipo']}]", f"{URL_APUC}/home/item.html?id={iid}"))
for iid, r in FONTES_ENG.items():
    if not r["titulo"].startswith("("):
        LINKS.append((f"Engemap — {r['titulo']} [{r['tipo']}]", f"{URL_ENGEMAP}/home/item.html?id={iid}"))

log("\n  ===== LINKS =====")
for nome, url in LINKS:
    log(f"   {nome}\n     {url}")
log(f"\n  Saída completa: {ARQ_TXT}")
log("  Próximo passo: arraste esse .txt para o chat.")
log("\n✔ CÉLULA 6 concluída com sucesso — inventário finalizado (nada foi alterado nos portais).")
